In [2]:
# Install XGBoost if not present (usually pre-installed on Colab)
# !pip install xgboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import matplotlib.pyplot as plt

# 1. Dataset Acquisition
print("Downloading and loading dataset...")
url = "https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv"
df = pd.read_csv(url)

# 2. Data Preprocessing & Feature Engineering
print("Preprocessing data...")
# Drop non-predictive columns like date and random variables (rv1, rv2)
df = df.drop(columns=['date', 'rv1', 'rv2'])

# Separate features (X) and target (y - Appliances energy consumption)
X = df.drop(columns=['Appliances'])
y = df['Appliances']

# Normalize features
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Helper function for evaluation metrics
def evaluate_model(y_true, y_pred, model_name):
    y_true_inv = scaler_y.inverse_transform(y_true.reshape(-1, 1))
    y_pred_inv = scaler_y.inverse_transform(y_pred.reshape(-1, 1))

    rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
    mae = mean_absolute_error(y_true_inv, y_pred_inv)

    # Avoid division by zero for MAPE
    y_true_inv_safe = np.where(y_true_inv == 0, 1e-10, y_true_inv)
    mape = np.mean(np.abs((y_true_inv - y_pred_inv) / y_true_inv_safe)) * 100

    r2 = r2_score(y_true_inv, y_pred_inv)

    return {"Model": model_name, "RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2 Score": r2}

results = []

# 3. Model Implementation

# A. Linear Regression (Baseline)
print("Training Linear Regression...")
lr = LinearRegression()
lr.fit(X_train, y_train.ravel())
lr_pred = lr.predict(X_test)
results.append(evaluate_model(y_test, lr_pred, "Linear Regression"))

# B. Random Forest
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train.ravel())
rf_pred = rf.predict(X_test)
results.append(evaluate_model(y_test, rf_pred, "Random Forest"))

# C. XGBoost
print("Training XGBoost...")
xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train.ravel())
xgb_pred = xgb.predict(X_test)
results.append(evaluate_model(y_test, xgb_pred, "XGBoost"))

# D. LSTM
print("Training LSTM...")
# Reshape data for LSTM [samples, time steps, features]
X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

lstm = Sequential([
    LSTM(50, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')
lstm.fit(X_train_lstm, y_train, epochs=20, batch_size=32, verbose=0) # verbose=0 to hide training logs
lstm_pred = lstm.predict(X_test_lstm)
results.append(evaluate_model(y_test, lstm_pred, "LSTM"))

# 4. Results & Comparison Table
results_df = pd.DataFrame(results)
print("\n--- Final Comparison Table ---")
print(results_df.to_markdown(index=False))

Preprocessing data...
Training Linear Regression...
Training Random Forest...
Training XGBoost...
Training LSTM...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


124/124 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

--- Final Comparison Table ---
| Model             |    RMSE |     MAE |   MAPE (%) |   R2 Score |
|:------------------|--------:|--------:|-----------:|-----------:|
| Linear Regression | 91.1717 | 52.5471 |    62.9209 |   0.169362 |
| Random Forest     | 67.8865 | 32.0357 |    32.8017 |   0.539469 |
| XGBoost           | 74.8267 | 38.2541 |    41.582  |   0.440493 |
| LSTM              | 88.6146 | 47.5254 |    49.8915 |   0.215301 |
